In [0]:
import os
import sys

from pyspark.sql import SparkSession
from pyspark.sql.window import Window

import pyspark.sql.functions as f
import pyspark.sql.types as t

In [0]:
dbutils.widgets.dropdown("position_filter", "WR", ["WR", "RB", "TE", "QB"])

In [0]:
# load parquet files into dataframes
df_25 = spark.read.parquet("/Volumes/workspace/default/fantasy_data/season_25/stats_player_reg_2025.parquet")
df_24 = spark.read.parquet("/Volumes/workspace/default/fantasy_data/season_24/stats_player_reg_2024.parquet")
df_23 = spark.read.parquet("/Volumes/workspace/default/fantasy_data/season_23/stats_player_reg_2023.parquet")
df_22 = spark.read.parquet("/Volumes/workspace/default/fantasy_data/season_22/stats_player_reg_2022.parquet")
# df_25.columns

In [0]:
# create union for all dataframes and add hppr files
total_df = df_25.union(df_24).union(df_23).union(df_22)
total_df = total_df.withColumn("hppr_points", (f.col("fantasy_points_ppr")+f.col("fantasy_points"))/2
                               ).withColumn("hppr_ppg", ((f.col("fantasy_points_ppr")+f.col("fantasy_points"))/2) / f.col("games"))
total_df = total_df.orderBy("hppr_ppg", ascending=False)

In [0]:
# create temporary view for SQL
total_df.createOrReplaceTempView("total_sql")

In [0]:
%sql
-- group WRs and find all time PPG

with players_grouped as (
    select
        player_display_name,
        position
    from
        total_sql
    group by player_display_name, position
)

select 
    pg.player_display_name, 
    pg.position,
    MAX(recent_team),
    ROUND(SUM(hppr_points) / SUM(games), 2) as avg_season_ppg,
    ROUND(SUM(receiving_yards) / SUM(games), 2) as avg_rec_ypg,
    ROUND(SUM(receiving_tds) / SUM(games), 2) as avg_rec_tds_pg,
    ROUND(SUM(rushing_yards) / SUM(games), 2) as avg_rush_ypg,
    ROUND(SUM(rushing_tds) / SUM(games), 2) as avg_rush_tds_pg,
    ROUND(SUM(passing_yards) / SUM(games), 2) as avg_pass_ypg,
    ROUND(SUM(passing_tds) / SUM(games), 2) as avg_pass_tds_pg
FROM players_grouped pg
JOIN total_sql t ON t.player_display_name = pg.player_display_name AND t.position = pg.position
WHERE t.games > 7 AND pg.position == :position_filter
GROUP BY ALL
ORDER BY avg_season_ppg DESC